# Preparação das áreas ardidas do ICNF

Este notebook reproduz os produtos para os cenários **C1–C4**:

In [ ]:
import sys
from glob import glob
from pathlib import Path

sys.path.append("/code/scripts")

from aoi_utils import check_alignment
from burned_area_utils import (
    align_outputs,
    create_binary_raster,
    create_count_raster,
    prepare_icnf
)

## Configuração

In [ ]:
area = "centro"
validation_year = 2025

raw_shps = sorted(glob("/code/data/raw/area_ardida/icnf/*/*.shp"))
base = f"/code/data/processed/{area}"
aa_dir = f"{base}/area_ardida/icnf"

aoi = f"{base}/aoi/{area}.shp"
reference = f"{base}/reference/target_grid_25m.tif"

out_year = f"{aa_dir}/yearly_vector"
out_aoi = f"{aa_dir}/yearly_aoi"
out_train = f"{aa_dir}/train"
out_valid = f"{aa_dir}/valid"
out_count = f"{aa_dir}/raster_count"
out_binary = f"{aa_dir}/raster_binary"
out_temp = f"/code/data/scratch/tmp_periods/{area}/area_ardida"

## Preparação dos vetores anuais

In [ ]:
summary = prepare_icnf(
    raw_shps=raw_shps,
    aoi=aoi,
    reference=reference,
    yearly_folder=out_year,
    clipped_folder=out_aoi,
    train_folder=out_train,
    valid_folder=out_valid,
    validation_year=validation_year
)

summary

In [ ]:
print("Anuais separados:", len(list(Path(out_year).glob("aa_*.shp"))))
print("Anuais recortados:", len(list(Path(out_aoi).glob("aa_*.shp"))))
print("Treino:", len(list(Path(out_train).glob("aa_*.shp"))))
print("Validação:", len(list(Path(out_valid).glob("aa_*.shp"))))

## Períodos temporais de C1–C4

In [ ]:
periods = {
    "1995_2024": (range(1995, 2025), out_train),
    "1995_2006": (range(1995, 2007), out_aoi),
    "2007_2009": (range(2007, 2010), out_aoi),
    "2010_2014": (range(2010, 2015), out_aoi),
    "2015_2017": (range(2015, 2018), out_aoi),
    "2018_2024": (range(2018, 2025), out_aoi),
    "2008_2024": (range(2008, 2025), out_train),
    "2008_2009": (range(2008, 2010), out_aoi),
    "2025": ([2025], out_valid)
}

## Rasters de contagem

In [ ]:
count_rasters = []

for name, (years, folder) in periods.items():
    output = f"{out_count}/rst_ba_{name}.tif"

    create_count_raster(
        years=years,
        source_folder=folder,
        temp_folder=f"{out_temp}/{name}",
        reference=reference,
        output=output
    )

    count_rasters.append(output)
    print("Criado:", output)

## Rasters binários

In [ ]:
binary_periods = ["1995_2024", "2025", "2008_2024"]
binary_rasters = []

for name in binary_periods:
    source = f"{out_count}/rst_ba_{name}.tif"
    output = f"{out_binary}/rst_ba_{name}_bin.tif"

    create_binary_raster(source, output)
    binary_rasters.append(output)
    print("Criado:", output)

## Alinhamento e verificação final

In [ ]:
rasters = count_rasters + binary_rasters

align_outputs(rasters, reference, aoi)
check_alignment(rasters, reference)

In [ ]:
expected_count = {
    "rst_ba_1995_2024.tif",
    "rst_ba_1995_2006.tif",
    "rst_ba_2007_2009.tif",
    "rst_ba_2010_2014.tif",
    "rst_ba_2015_2017.tif",
    "rst_ba_2018_2024.tif",
    "rst_ba_2008_2024.tif",
    "rst_ba_2008_2009.tif",
    "rst_ba_2025.tif"
}

expected_binary = {
    "rst_ba_1995_2024_bin.tif",
    "rst_ba_2008_2024_bin.tif",
    "rst_ba_2025_bin.tif"
}

assert {Path(path).name for path in count_rasters} == expected_count
assert {Path(path).name for path in binary_rasters} == expected_binary
assert len(list(Path(out_year).glob("aa_*.shp"))) == 51
assert len(list(Path(out_aoi).glob("aa_*.shp"))) == 51
assert len(list(Path(out_train).glob("aa_*.shp"))) == 50
assert len(list(Path(out_valid).glob("aa_*.shp"))) == 1

print("Todos os produtos foram criados.")